In [4]:
!pip install svgpathtools
!pip install svg.path

     -------------------------------------- 40.9/40.9 kB 957.8 kB/s eta 0:00:00


In [5]:
import os
import sys
import math
import re
import shutil  # <--- Added for copying files
import numpy as np
import xml.etree.ElementTree as ET

# --- Library Imports ---
from svgpathtools import svg2paths, wsvg
from svgpathtools import Path as ToolsPath, Line as ToolsLine, CubicBezier as ToolsCubic, QuadraticBezier as ToolsQuad, Arc as ToolsArc
from svg.path import parse_path, Line as SvgLine, CubicBezier as SvgCubic

# ==============================================================================
# PART 1: GEOMETRIC ROTATION & FILE MANAGEMENT
# ==============================================================================

def rotate_point(x, y, cx, cy, angle_deg):
    theta = math.radians(-angle_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    x -= cx
    y -= cy
    xr = cos_t * x - sin_t * y + cx
    yr = sin_t * x + cos_t * y + cy
    return xr, yr

def rotate_path_obj(path, cx, cy, angle_deg):
    new_segments = []
    for seg in path:
        if isinstance(seg, ToolsLine):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_deg)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_deg)
            new_segments.append(ToolsLine(complex(*s), complex(*e)))
        elif isinstance(seg, ToolsCubic):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_deg)
            c1 = rotate_point(seg.control1.real, seg.control1.imag, cx, cy, angle_deg)
            c2 = rotate_point(seg.control2.real, seg.control2.imag, cx, cy, angle_deg)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_deg)
            new_segments.append(ToolsCubic(complex(*s), complex(*c1), complex(*c2), complex(*e)))
        elif isinstance(seg, ToolsQuad):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_deg)
            c = rotate_point(seg.control.real, seg.control.imag, cx, cy, angle_deg)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_deg)
            new_segments.append(ToolsQuad(complex(*s), complex(*c), complex(*e)))
        elif isinstance(seg, ToolsArc):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_deg)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_deg)
            new_segments.append(ToolsArc(complex(*s), seg.radius, seg.rotation + angle_deg, 
                                         seg.large_arc, seg.sweep, complex(*e)))
    return ToolsPath(*new_segments)

def extract_viewbox(attributes):
    for attr in attributes:
        if "viewBox" in attr:
            return list(map(float, attr["viewBox"].split()))
    for attr in attributes:
        if "width" in attr and "height" in attr:
            w = float(attr["width"].replace("px", ""))
            h = float(attr["height"].replace("px", ""))
            return [0.0, 0.0, w, h]
    return [0.0, 0.0, 255.0, 255.0]

def create_rotated_iso_file(input_path, output_path, angle=120):
    """Reads the ISO SVG, rotates it, and saves it."""
    if not os.path.exists(input_path):
        print(f"  [!] Source not found: {input_path}")
        return False
    try:
        paths, attributes = svg2paths(input_path)
        vx, vy, vw, vh = extract_viewbox(attributes)
        cx, cy = vx + vw / 2.0, vy + vh / 2.0
        rotated_paths = [rotate_path_obj(p, cx, cy, angle) for p in paths]
        wsvg(
            rotated_paths, filename=output_path,
            svg_attributes={"viewBox": f"{vx} {vy} {vw} {vh}", "width": str(vw), "height": str(vh), "xmlns": "http://www.w3.org/2000/svg"}
        )
        return True
    except Exception as e:
        print(f"  [!] Error rotating ISO: {e}")
        return False

# ==============================================================================
# PART 2: VECTORIZATION & NPY CREATION
# ==============================================================================

SVG_SOS_IDX, SVG_EOS_IDX, SVG_L_IDX, SVG_C_IDX = 0, 1, 2, 3
PAD_VAL = -1.0
TARGET_MIN, TARGET_MAX = 0.0, 255.0
TARGET_RANGE = TARGET_MAX - TARGET_MIN
N_COMMANDS = 100
N_PARAMS = 10 

def get_sos_vec(view_label):
    return np.array([view_label, SVG_SOS_IDX, *([PAD_VAL] * 8)], dtype=np.float32)

def get_eos_vec(view_label):
    return np.array([view_label, SVG_EOS_IDX, *([PAD_VAL] * 8)], dtype=np.float32)

def normalize_coords(x, y, vb):
    min_x, min_y, vb_width, vb_height = vb
    if vb_width == 0 or vb_height == 0: return (TARGET_MIN, TARGET_MIN)
    norm_x = ((x - min_x) / vb_width) * TARGET_RANGE + TARGET_MIN
    norm_y = ((y - min_y) / vb_height) * TARGET_RANGE + TARGET_MIN
    return np.floor(norm_x), np.floor(norm_y)

def format_line_vec(segment, view_label, vb):
    x1, y1 = normalize_coords(segment.start.real, segment.start.imag, vb)
    x2, y2 = normalize_coords(segment.end.real, segment.end.imag, vb)
    return np.array([view_label, SVG_L_IDX, x1, y1, PAD_VAL, PAD_VAL, PAD_VAL, PAD_VAL, x2, y2], dtype=np.float32)

def format_bezier_vec(segment, view_label, vb):
    x1, y1 = normalize_coords(segment.start.real, segment.start.imag, vb)
    cx1, cy1 = normalize_coords(segment.control1.real, segment.control1.imag, vb)
    cx2, cy2 = normalize_coords(segment.control2.real, segment.control2.imag, vb)
    x2, y2 = normalize_coords(segment.end.real, segment.end.imag, vb)
    return np.array([view_label, SVG_C_IDX, x1, y1, cx1, cy1, cx2, cy2, x2, y2], dtype=np.float32)

def get_viewbox_xml(root):
    viewBox_str = root.get('viewBox')
    if viewBox_str:
        vb = [float(v) for v in re.findall(r"[-+]?\d*\.?\d+", viewBox_str)]
        if len(vb) == 4: return vb
    width = root.get('width')
    height = root.get('height')
    if width and height:
        try:
            w = float(re.findall(r"[-+]?\d*\.?\d+", width)[0])
            h = float(re.findall(r"[-+]?\d*\.?\d+", height)[0])
            return [0.0, 0.0, w, h]
        except: pass
    return [0.0, 0.0, 255.0, 255.0]

def convert_svg_to_sequence(svg_file_path, view_label):
    command_sequence = []
    try:
        ET.register_namespace('', "http://www.w3.org/2000/svg")
        tree = ET.parse(svg_file_path)
        root = tree.getroot()
        vb = get_viewbox_xml(root)

        command_sequence.append(get_sos_vec(view_label))
        
        namespaces = {'svg': 'http://www.w3.org/2000/svg'}
        elements = root.findall('.//svg:path', namespaces) + root.findall('.//svg:line', namespaces)
        if not elements:
            elements = root.findall('.//{http://www.w3.org/2000/svg}path') + root.findall('.//{http://www.w3.org/2000/svg}line')

        for elem in elements:
            if elem.tag.endswith('line'):
                x1, y1 = float(elem.get('x1', 0)), float(elem.get('y1', 0))
                x2, y2 = float(elem.get('x2', 0)), float(elem.get('y2', 0))
                mock = type('obj', (object,), {'start': complex(x1, y1), 'end': complex(x2, y2)})
                command_sequence.append(format_line_vec(mock, view_label, vb))
            elif elem.tag.endswith('path'):
                d_string = elem.get('d')
                if d_string:
                    for seg in parse_path(d_string):
                        if isinstance(seg, SvgLine):
                            command_sequence.append(format_line_vec(seg, view_label, vb))
                        elif isinstance(seg, SvgCubic):
                            command_sequence.append(format_bezier_vec(seg, view_label, vb))
                            
        command_sequence.append(get_eos_vec(view_label))
        return np.array(command_sequence, dtype=np.float32)
    except Exception as e:
        print(f"  [!] Error parsing {os.path.basename(svg_file_path)}: {e}")
        return None

def pad_sequence(sequence, view_label, target_length=N_COMMANDS):
    if sequence is None: sequence = np.empty((0, N_PARAMS), dtype=np.float32)
    current_length = sequence.shape[0]
    if current_length > target_length:
        padded = sequence[:target_length]
        padded[-1] = get_eos_vec(view_label)
    elif current_length < target_length:
        pad_vec = get_eos_vec(view_label)
        padding = np.tile(pad_vec, (target_length - current_length, 1))
        padded = np.vstack((sequence, padding))
    else:
        padded = sequence
    return padded

# ==============================================================================
# PART 3: MAIN PIPELINE
# ==============================================================================

def process_single_sample(folder_path, output_npy_path):
    base_name = os.path.basename(folder_path)
    print(f"Processing: {base_name}")

    # 1. Define Source Paths
    src_front = os.path.join(folder_path, f"{base_name}_Front.svg")
    src_top   = os.path.join(folder_path, f"{base_name}_Top.svg")
    src_right = os.path.join(folder_path, f"{base_name}_Right.svg")
    src_iso   = os.path.join(folder_path, f"{base_name}_FrontTopRight.svg")

    # 2. Define Final Output Paths (We will save ALL 4 files here)
    final_front = os.path.join(folder_path, f"{base_name}_Front_final.svg")
    final_right = os.path.join(folder_path, f"{base_name}_Right_final.svg")
    final_top   = os.path.join(folder_path, f"{base_name}_Top_final.svg")
    final_iso   = os.path.join(folder_path, f"{base_name}_FrontTopRight_final.svg")

    print("  [~] Preparing Final SVGs...")

    # 3. Create Final SVGs
    # A. Rotate Iso (Geometry Change)
    create_rotated_iso_file(src_iso, final_iso, angle=120)

    # B. Copy others (No Geometry Change, just saving copy)
    for src, dst in [(src_front, final_front), (src_right, final_right), (src_top, final_top)]:
        if os.path.exists(src):
            shutil.copy(src, dst)

    # 4. Generate NPY from the FINAL files
    # Order: Front(0), Right(1), Top(2), Iso(3)
    processing_queue = [
        ("Front", final_front, 0),
        ("Right", final_right, 1), 
        ("Top",   final_top,   2),
        ("Iso",   final_iso,   3)
    ]

    all_sequences = []

    for name, fpath, label in processing_queue:
        if not os.path.exists(fpath):
            print(f"  [!] Missing file: {os.path.basename(fpath)}")
            raw_seq = None
        else:
            raw_seq = convert_svg_to_sequence(fpath, label)
        
        padded_seq = pad_sequence(raw_seq, label)
        all_sequences.append(padded_seq)

    # 5. Stack and Save NPY
    try:
        final_stack = np.vstack(all_sequences)
        np.save(output_npy_path, final_stack)
        print(f"  [+] Saved NPY: {os.path.basename(output_npy_path)}")
        print(f"  [+] All 4 final SVGs are in: {folder_path}")
    except Exception as e:
        print(f"  [!] Stacking error: {e}")

if __name__ == "__main__":
    INPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw\0001\00019749"
    
    if "svg_raw" in INPUT_DIR:
        base_path = INPUT_DIR.split(r"\svg_raw")[0]
        sub_path = INPUT_DIR.split(r"\svg_raw")[1]
        OUTPUT_FILE = os.path.join(base_path, "svg_vec_new" + sub_path + ".npy")
    else:
        OUTPUT_FILE = os.path.join(INPUT_DIR, "output.npy")
        
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    process_single_sample(INPUT_DIR, OUTPUT_FILE)

Processing: 00019749
  [~] Preparing Final SVGs...
  [+] Saved NPY: 00019749.npy
  [+] All 4 final SVGs are in: C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw\0001\00019749


In [12]:
import os
import sys
import math
import re
import numpy as np
import xml.etree.ElementTree as ET

# --- Library Imports ---
from svgpathtools import svg2paths, wsvg
from svgpathtools import Path as ToolsPath, Line as ToolsLine, CubicBezier as ToolsCubic, QuadraticBezier as ToolsQuad, Arc as ToolsArc
from svg.path import parse_path, Line as SvgLine, CubicBezier as SvgCubic

# ==============================================================================
# PART 1: GEOMETRIC ROTATION LOGIC
# ==============================================================================

def rotate_point(x, y, cx, cy, angle_degrees_ccw):
    """
    Rotates a point (x,y) around (cx,cy) by angle_degrees_ccw.
    Positive angle = Counter-Clockwise.
    """
    theta = math.radians(angle_degrees_ccw)
    # Standard 2D rotation matrix for Counter-Clockwise
    # x' = x cos(t) - y sin(t)
    # y' = x sin(t) + y cos(t)
    # Note: SVG Y-axis is down, but the math holds relative to the center.
    
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    
    x -= cx
    y -= cy
    
    xr = cos_t * x - sin_t * y + cx
    yr = sin_t * x + cos_t * y + cy
    return xr, yr

def rotate_path_obj(path, cx, cy, angle_ccw):
    new_segments = []
    for seg in path:
        if isinstance(seg, ToolsLine):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            new_segments.append(ToolsLine(complex(*s), complex(*e)))
        
        elif isinstance(seg, ToolsCubic):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            c1 = rotate_point(seg.control1.real, seg.control1.imag, cx, cy, angle_ccw)
            c2 = rotate_point(seg.control2.real, seg.control2.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            new_segments.append(ToolsCubic(complex(*s), complex(*c1), complex(*c2), complex(*e)))
            
        elif isinstance(seg, ToolsQuad):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            c = rotate_point(seg.control.real, seg.control.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            new_segments.append(ToolsQuad(complex(*s), complex(*c), complex(*e)))
            
        elif isinstance(seg, ToolsArc):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            # In SVG coordinate space (y-down), positive rotation is usually clockwise visually.
            # But since we are calculating coord transforms manually, we just adjust the rotation param.
            # For standard math CCW, we subtract the angle because SVG 'rotation' attribute is often CW.
            # We will approximate by subtracting angle_ccw.
            new_segments.append(ToolsArc(complex(*s), seg.radius, seg.rotation - angle_ccw, 
                                         seg.large_arc, seg.sweep, complex(*e)))
            
    return ToolsPath(*new_segments)

def extract_viewbox(attributes):
    for attr in attributes:
        if "viewBox" in attr:
            return list(map(float, attr["viewBox"].split()))
    for attr in attributes:
        if "width" in attr and "height" in attr:
            w = float(attr["width"].replace("px", ""))
            h = float(attr["height"].replace("px", ""))
            return [0.0, 0.0, w, h]
    return [0.0, 0.0, 255.0, 255.0]

def create_rotated_file(input_path, output_path, angle_ccw):
    """
    Reads an SVG, rotates it by angle_ccw (Counter-Clockwise), and saves it.
    """
    if not os.path.exists(input_path):
        print(f"  [!] Missing Source: {os.path.basename(input_path)}")
        return False

    try:
        paths, attributes = svg2paths(input_path)
        vx, vy, vw, vh = extract_viewbox(attributes)
        
        # Center of rotation
        cx, cy = vx + vw / 2.0, vy + vh / 2.0

        rotated_paths = [rotate_path_obj(p, cx, cy, angle_ccw) for p in paths]

        # Note: If rotating by 90 degrees, width and height usually swap.
        # But for simplicity in a square canvas (common in AI datasets), we keep viewbox same.
        # If your canvas is rectangular, you might need to swap vw/vh in the viewbox below.
        
        wsvg(
            rotated_paths,
            filename=output_path,
            svg_attributes={
                "viewBox": f"{vx} {vy} {vw} {vh}",
                "width": str(vw),
                "height": str(vh),
                "xmlns": "http://www.w3.org/2000/svg"
            }
        )
        return True
    except Exception as e:
        print(f"  [!] Error processing {os.path.basename(input_path)}: {e}")
        return False

# ==============================================================================
# PART 2: VECTORIZATION HELPERS
# ==============================================================================

SVG_SOS_IDX, SVG_EOS_IDX, SVG_L_IDX, SVG_C_IDX = 0, 1, 2, 3
PAD_VAL = -1.0
TARGET_MIN, TARGET_MAX = 0.0, 255.0
TARGET_RANGE = TARGET_MAX - TARGET_MIN
N_COMMANDS = 100
N_PARAMS = 10 

def get_sos_vec(view_label):
    return np.array([view_label, SVG_SOS_IDX, *([PAD_VAL] * 8)], dtype=np.float32)

def get_eos_vec(view_label):
    return np.array([view_label, SVG_EOS_IDX, *([PAD_VAL] * 8)], dtype=np.float32)

def normalize_coords(x, y, vb):
    min_x, min_y, vb_width, vb_height = vb
    if vb_width == 0 or vb_height == 0: return (TARGET_MIN, TARGET_MIN)
    norm_x = ((x - min_x) / vb_width) * TARGET_RANGE + TARGET_MIN
    norm_y = ((y - min_y) / vb_height) * TARGET_RANGE + TARGET_MIN
    return np.floor(norm_x), np.floor(norm_y)

def format_line_vec(segment, view_label, vb):
    x1, y1 = normalize_coords(segment.start.real, segment.start.imag, vb)
    x2, y2 = normalize_coords(segment.end.real, segment.end.imag, vb)
    return np.array([view_label, SVG_L_IDX, x1, y1, PAD_VAL, PAD_VAL, PAD_VAL, PAD_VAL, x2, y2], dtype=np.float32)

def format_bezier_vec(segment, view_label, vb):
    x1, y1 = normalize_coords(segment.start.real, segment.start.imag, vb)
    cx1, cy1 = normalize_coords(segment.control1.real, segment.control1.imag, vb)
    cx2, cy2 = normalize_coords(segment.control2.real, segment.control2.imag, vb)
    x2, y2 = normalize_coords(segment.end.real, segment.end.imag, vb)
    return np.array([view_label, SVG_C_IDX, x1, y1, cx1, cy1, cx2, cy2, x2, y2], dtype=np.float32)

def get_viewbox_xml(root):
    viewBox_str = root.get('viewBox')
    if viewBox_str:
        vb = [float(v) for v in re.findall(r"[-+]?\d*\.?\d+", viewBox_str)]
        if len(vb) == 4: return vb
    width = root.get('width')
    height = root.get('height')
    if width and height:
        try:
            w = float(re.findall(r"[-+]?\d*\.?\d+", width)[0])
            h = float(re.findall(r"[-+]?\d*\.?\d+", height)[0])
            return [0.0, 0.0, w, h]
        except: pass
    return [0.0, 0.0, 255.0, 255.0]

def convert_svg_to_sequence(svg_file_path, view_label):
    command_sequence = []
    try:
        ET.register_namespace('', "http://www.w3.org/2000/svg")
        tree = ET.parse(svg_file_path)
        root = tree.getroot()
        vb = get_viewbox_xml(root)

        command_sequence.append(get_sos_vec(view_label))
        
        namespaces = {'svg': 'http://www.w3.org/2000/svg'}
        elements = root.findall('.//svg:path', namespaces) + root.findall('.//svg:line', namespaces)
        if not elements:
            elements = root.findall('.//{http://www.w3.org/2000/svg}path') + root.findall('.//{http://www.w3.org/2000/svg}line')

        for elem in elements:
            if elem.tag.endswith('line'):
                x1, y1 = float(elem.get('x1', 0)), float(elem.get('y1', 0))
                x2, y2 = float(elem.get('x2', 0)), float(elem.get('y2', 0))
                mock = type('obj', (object,), {'start': complex(x1, y1), 'end': complex(x2, y2)})
                command_sequence.append(format_line_vec(mock, view_label, vb))
            elif elem.tag.endswith('path'):
                d_string = elem.get('d')
                if d_string:
                    for seg in parse_path(d_string):
                        if isinstance(seg, SvgLine):
                            command_sequence.append(format_line_vec(seg, view_label, vb))
                        elif isinstance(seg, SvgCubic):
                            command_sequence.append(format_bezier_vec(seg, view_label, vb))
                            
        command_sequence.append(get_eos_vec(view_label))
        return np.array(command_sequence, dtype=np.float32)
    except Exception as e:
        # print(f"  [!] Error parsing {os.path.basename(svg_file_path)}: {e}")
        return None

def pad_sequence(sequence, view_label, target_length=N_COMMANDS):
    if sequence is None: sequence = np.empty((0, N_PARAMS), dtype=np.float32)
    current_length = sequence.shape[0]
    if current_length > target_length:
        padded = sequence[:target_length]
        padded[-1] = get_eos_vec(view_label)
    elif current_length < target_length:
        pad_vec = get_eos_vec(view_label)
        padding = np.tile(pad_vec, (target_length - current_length, 1))
        padded = np.vstack((sequence, padding))
    else:
        padded = sequence
    return padded

# ==============================================================================
# PART 3: MAIN PIPELINE
# ==============================================================================

def process_folder(folder_path, output_npy_path):
    base_name = os.path.basename(folder_path)
    print(f"Processing: {base_name}")

    # --- 1. DEFINE OLD FILES (INPUTS) ---
    old_front = os.path.join(folder_path, f"{base_name}_Front.svg")
    old_top   = os.path.join(folder_path, f"{base_name}_Top.svg")
    old_right = os.path.join(folder_path, f"{base_name}_Right.svg")
    old_iso   = os.path.join(folder_path, f"{base_name}_FrontTopRight.svg")

    # --- 2. DEFINE NEW FILES (OUTPUTS) ---
    new_front = os.path.join(folder_path, f"{base_name}_Front_final.svg")
    new_top   = os.path.join(folder_path, f"{base_name}_Top_final.svg")
    new_right = os.path.join(folder_path, f"{base_name}_Right_final.svg")
    new_iso   = os.path.join(folder_path, f"{base_name}_FrontTopRight_final.svg")

    print("  [~] Performing Geometric Rotations...")

    # --- 3. APPLY ROTATIONS ---
    # New Front = Old Top (Rotated 90 CCW)
    create_rotated_file(old_top, new_front, angle_ccw=90)

    # New Top = Old Right (Rotated 90 CCW)
    create_rotated_file(old_right, new_top, angle_ccw=90)

    # New Side/Right = Old Front (Rotated 90 CCW)
    create_rotated_file(old_front, new_right, angle_ccw=90)

    # New Iso = Old Iso (Rotated 120 Clockwise -> -120 CCW)
    create_rotated_file(old_iso, new_iso, angle_ccw=-120)

    print("  [+] All 4 final SVGs saved.")

    # --- 4. CREATE NPY STACK ---
    # Order: New Front (0), New Side (1), New Top (2), New Iso (3)
    
    views_config = [
        (new_front, 0),
        (new_right, 1),
        (new_top,   2),
        (new_iso,   3)
    ]

    all_sequences = []

    for fpath, label in views_config:
        raw_seq = convert_svg_to_sequence(fpath, label)
        padded_seq = pad_sequence(raw_seq, label)
        all_sequences.append(padded_seq)

    try:
        final_stack = np.vstack(all_sequences)
        np.save(output_npy_path, final_stack)
        print(f"  [+] Saved NPY: {os.path.basename(output_npy_path)}")
        print(f"  [i] Shape: {final_stack.shape}")
    except Exception as e:
        print(f"  [!] Stacking error: {e}")

if __name__ == "__main__":
    # CONFIGURATION
    INPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw\0000\00000660"
    
    # Auto-output path setup
    if "svg_raw" in INPUT_DIR:
        base_path = INPUT_DIR.split(r"\svg_raw")[0]
        sub_path = INPUT_DIR.split(r"\svg_raw")[1]
        OUTPUT_FILE = os.path.join(base_path, "svg_vec_new" + sub_path + ".npy")
    else:
        OUTPUT_FILE = os.path.join(INPUT_DIR, "output.npy")
        
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    process_folder(INPUT_DIR, OUTPUT_FILE)

Processing: 00000660
  [~] Performing Geometric Rotations...
  [+] All 4 final SVGs saved.
  [+] Saved NPY: 00000660.npy
  [i] Shape: (400, 10)


In [ ]:
import os
import sys
import math
import re
import numpy as np
import xml.etree.ElementTree as ET

# --- Library Imports ---
from svgpathtools import svg2paths, wsvg
from svgpathtools import Path as ToolsPath, Line as ToolsLine, CubicBezier as ToolsCubic, QuadraticBezier as ToolsQuad, Arc as ToolsArc
from svg.path import parse_path, Line as SvgLine, CubicBezier as SvgCubic

# ==============================================================================
# PART 1: GEOMETRIC ROTATION LOGIC
# ==============================================================================

def rotate_point(x, y, cx, cy, angle_degrees_ccw):
    """
    Rotates a point (x,y) around (cx,cy) by angle_degrees_ccw.
    Positive angle = Counter-Clockwise.
    """
    theta = math.radians(angle_degrees_ccw)
    # Standard 2D rotation matrix for Counter-Clockwise
    # x' = x cos(t) - y sin(t)
    # y' = x sin(t) + y cos(t)
    # Note: SVG Y-axis is down, but the math holds relative to the center.
    
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    
    x -= cx
    y -= cy
    
    xr = cos_t * x - sin_t * y + cx
    yr = sin_t * x + cos_t * y + cy
    return xr, yr

def rotate_path_obj(path, cx, cy, angle_ccw):
    new_segments = []
    for seg in path:
        if isinstance(seg, ToolsLine):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            new_segments.append(ToolsLine(complex(*s), complex(*e)))
        
        elif isinstance(seg, ToolsCubic):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            c1 = rotate_point(seg.control1.real, seg.control1.imag, cx, cy, angle_ccw)
            c2 = rotate_point(seg.control2.real, seg.control2.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            new_segments.append(ToolsCubic(complex(*s), complex(*c1), complex(*c2), complex(*e)))
            
        elif isinstance(seg, ToolsQuad):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            c = rotate_point(seg.control.real, seg.control.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            new_segments.append(ToolsQuad(complex(*s), complex(*c), complex(*e)))
            
        elif isinstance(seg, ToolsArc):
            s = rotate_point(seg.start.real, seg.start.imag, cx, cy, angle_ccw)
            e = rotate_point(seg.end.real, seg.end.imag, cx, cy, angle_ccw)
            # In SVG coordinate space (y-down), positive rotation is usually clockwise visually.
            # But since we are calculating coord transforms manually, we just adjust the rotation param.
            # For standard math CCW, we subtract the angle because SVG 'rotation' attribute is often CW.
            # We will approximate by subtracting angle_ccw.
            new_segments.append(ToolsArc(complex(*s), seg.radius, seg.rotation - angle_ccw, 
                                         seg.large_arc, seg.sweep, complex(*e)))
            
    return ToolsPath(*new_segments)

def extract_viewbox(attributes):
    for attr in attributes:
        if "viewBox" in attr:
            return list(map(float, attr["viewBox"].split()))
    for attr in attributes:
        if "width" in attr and "height" in attr:
            w = float(attr["width"].replace("px", ""))
            h = float(attr["height"].replace("px", ""))
            return [0.0, 0.0, w, h]
    return [0.0, 0.0, 255.0, 255.0]

def create_rotated_file(input_path, output_path, angle_ccw):
    """
    Reads an SVG, rotates it by angle_ccw (Counter-Clockwise), and saves it.
    """
    if not os.path.exists(input_path):
        print(f"  [!] Missing Source: {os.path.basename(input_path)}")
        return False

    try:
        paths, attributes = svg2paths(input_path)
        vx, vy, vw, vh = extract_viewbox(attributes)
        
        # Center of rotation
        cx, cy = vx + vw / 2.0, vy + vh / 2.0

        rotated_paths = [rotate_path_obj(p, cx, cy, angle_ccw) for p in paths]

        # Note: If rotating by 90 degrees, width and height usually swap.
        # But for simplicity in a square canvas (common in AI datasets), we keep viewbox same.
        # If your canvas is rectangular, you might need to swap vw/vh in the viewbox below.
        
        wsvg(
            rotated_paths,
            filename=output_path,
            svg_attributes={
                "viewBox": f"{vx} {vy} {vw} {vh}",
                "width": str(vw),
                "height": str(vh),
                "xmlns": "http://www.w3.org/2000/svg"
            }
        )
        return True
    except Exception as e:
        print(f"  [!] Error processing {os.path.basename(input_path)}: {e}")
        return False

# ==============================================================================
# PART 2: VECTORIZATION HELPERS
# ==============================================================================

SVG_SOS_IDX, SVG_EOS_IDX, SVG_L_IDX, SVG_C_IDX = 0, 1, 2, 3
PAD_VAL = -1.0
TARGET_MIN, TARGET_MAX = 0.0, 255.0
TARGET_RANGE = TARGET_MAX - TARGET_MIN
N_COMMANDS = 100
N_PARAMS = 10 

def get_sos_vec(view_label):
    return np.array([view_label, SVG_SOS_IDX, *([PAD_VAL] * 8)], dtype=np.float32)

def get_eos_vec(view_label):
    return np.array([view_label, SVG_EOS_IDX, *([PAD_VAL] * 8)], dtype=np.float32)

def normalize_coords(x, y, vb):
    min_x, min_y, vb_width, vb_height = vb
    if vb_width == 0 or vb_height == 0: return (TARGET_MIN, TARGET_MIN)
    norm_x = ((x - min_x) / vb_width) * TARGET_RANGE + TARGET_MIN
    norm_y = ((y - min_y) / vb_height) * TARGET_RANGE + TARGET_MIN
    return np.floor(norm_x), np.floor(norm_y)

def format_line_vec(segment, view_label, vb):
    x1, y1 = normalize_coords(segment.start.real, segment.start.imag, vb)
    x2, y2 = normalize_coords(segment.end.real, segment.end.imag, vb)
    return np.array([view_label, SVG_L_IDX, x1, y1, PAD_VAL, PAD_VAL, PAD_VAL, PAD_VAL, x2, y2], dtype=np.float32)

def format_bezier_vec(segment, view_label, vb):
    x1, y1 = normalize_coords(segment.start.real, segment.start.imag, vb)
    cx1, cy1 = normalize_coords(segment.control1.real, segment.control1.imag, vb)
    cx2, cy2 = normalize_coords(segment.control2.real, segment.control2.imag, vb)
    x2, y2 = normalize_coords(segment.end.real, segment.end.imag, vb)
    return np.array([view_label, SVG_C_IDX, x1, y1, cx1, cy1, cx2, cy2, x2, y2], dtype=np.float32)

def get_viewbox_xml(root):
    viewBox_str = root.get('viewBox')
    if viewBox_str:
        vb = [float(v) for v in re.findall(r"[-+]?\d*\.?\d+", viewBox_str)]
        if len(vb) == 4: return vb
    width = root.get('width')
    height = root.get('height')
    if width and height:
        try:
            w = float(re.findall(r"[-+]?\d*\.?\d+", width)[0])
            h = float(re.findall(r"[-+]?\d*\.?\d+", height)[0])
            return [0.0, 0.0, w, h]
        except: pass
    return [0.0, 0.0, 255.0, 255.0]

def convert_svg_to_sequence(svg_file_path, view_label):
    command_sequence = []
    try:
        ET.register_namespace('', "http://www.w3.org/2000/svg")
        tree = ET.parse(svg_file_path)
        root = tree.getroot()
        vb = get_viewbox_xml(root)

        command_sequence.append(get_sos_vec(view_label))
        
        namespaces = {'svg': 'http://www.w3.org/2000/svg'}
        elements = root.findall('.//svg:path', namespaces) + root.findall('.//svg:line', namespaces)
        if not elements:
            elements = root.findall('.//{http://www.w3.org/2000/svg}path') + root.findall('.//{http://www.w3.org/2000/svg}line')

        for elem in elements:
            if elem.tag.endswith('line'):
                x1, y1 = float(elem.get('x1', 0)), float(elem.get('y1', 0))
                x2, y2 = float(elem.get('x2', 0)), float(elem.get('y2', 0))
                mock = type('obj', (object,), {'start': complex(x1, y1), 'end': complex(x2, y2)})
                command_sequence.append(format_line_vec(mock, view_label, vb))
            elif elem.tag.endswith('path'):
                d_string = elem.get('d')
                if d_string:
                    for seg in parse_path(d_string):
                        if isinstance(seg, SvgLine):
                            command_sequence.append(format_line_vec(seg, view_label, vb))
                        elif isinstance(seg, SvgCubic):
                            command_sequence.append(format_bezier_vec(seg, view_label, vb))
                            
        command_sequence.append(get_eos_vec(view_label))
        return np.array(command_sequence, dtype=np.float32)
    except Exception as e:
        # print(f"  [!] Error parsing {os.path.basename(svg_file_path)}: {e}")
        return None

def pad_sequence(sequence, view_label, target_length=N_COMMANDS):
    if sequence is None: sequence = np.empty((0, N_PARAMS), dtype=np.float32)
    current_length = sequence.shape[0]
    if current_length > target_length:
        padded = sequence[:target_length]
        padded[-1] = get_eos_vec(view_label)
    elif current_length < target_length:
        pad_vec = get_eos_vec(view_label)
        padding = np.tile(pad_vec, (target_length - current_length, 1))
        padded = np.vstack((sequence, padding))
    else:
        padded = sequence
    return padded

# ==============================================================================
# PART 3: MAIN PIPELINE
# ==============================================================================

def process_folder(folder_path, output_npy_path):
    base_name = os.path.basename(folder_path)
    print(f"Processing: {base_name}")

    # --- 1. DEFINE OLD FILES (INPUTS) ---
    old_front = os.path.join(folder_path, f"{base_name}_Front.svg")
    old_top   = os.path.join(folder_path, f"{base_name}_Top.svg")
    old_right = os.path.join(folder_path, f"{base_name}_Right.svg")
    old_iso   = os.path.join(folder_path, f"{base_name}_FrontTopRight.svg")

    # --- 2. DEFINE NEW FILES (OUTPUTS) ---
    new_front = os.path.join(folder_path, f"{base_name}_Front_final.svg")
    new_top   = os.path.join(folder_path, f"{base_name}_Top_final.svg")
    new_right = os.path.join(folder_path, f"{base_name}_Right_final.svg")
    new_iso   = os.path.join(folder_path, f"{base_name}_FrontTopRight_final.svg")

    print("  [~] Performing Geometric Rotations...")

    # --- 3. APPLY ROTATIONS ---
    # New Front = Old Top (Rotated 90 CCW)
    create_rotated_file(old_top, new_front, angle_ccw=90)

    # New Top = Old Right (Rotated 90 CCW)
    create_rotated_file(old_right, new_top, angle_ccw=90)

    # New Side/Right = Old Front (Rotated 90 CCW)
    create_rotated_file(old_front, new_right, angle_ccw=90)

    # New Iso = Old Iso (Rotated 120 Clockwise -> -120 CCW)
    create_rotated_file(old_iso, new_iso, angle_ccw=-120)

    print("  [+] All 4 final SVGs saved.")

    # --- 4. CREATE NPY STACK ---
    # Order: New Front (0), New Side (1), New Top (2), New Iso (3)
    
    views_config = [
        (new_front, 0),
        (new_right, 1),
        (new_top,   2),
        (new_iso,   3)
    ]

    all_sequences = []

    for fpath, label in views_config:
        raw_seq = convert_svg_to_sequence(fpath, label)
        padded_seq = pad_sequence(raw_seq, label)
        all_sequences.append(padded_seq)

    try:
        final_stack = np.vstack(all_sequences)
        np.save(output_npy_path, final_stack)
        print(f"  [+] Saved NPY: {os.path.basename(output_npy_path)}")
        print(f"  [i] Shape: {final_stack.shape}")
    except Exception as e:
        print(f"  [!] Stacking error: {e}")

if __name__ == "__main__":
    # CONFIGURATION
    INPUT_DIR = r"C:\Users\LEGION\Desktop\cad_project\DeepCAD\data2\svg_raw\0000\00000660"
    
    # Auto-output path setup
    if "svg_raw" in INPUT_DIR:
        base_path = INPUT_DIR.split(r"\svg_raw")[0]
        sub_path = INPUT_DIR.split(r"\svg_raw")[1]
        OUTPUT_FILE = os.path.join(base_path, "svg_vec_new" + sub_path + ".npy")
    else:
        OUTPUT_FILE = os.path.join(INPUT_DIR, "output.npy")
        
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    process_folder(INPUT_DIR, OUTPUT_FILE)

In [13]:
# Load and inspect the generated NPY stack
arr = np.load("C:\\Users\\LEGION\\Desktop\\cad_project\\DeepCAD\\data2\\svg_vec\\0000\\00000660.npy")
# print("Loaded shape:", arr.shape)
# print("First sequence (view 0) preview:\n", arr)

np.set_printoptions(threshold=arr.size, linewidth=200)

print("Loaded shape:", arr.shape)
print("Full array:\n", arr)

Loaded shape: (400, 10)
Full array:
 [[  0.   0.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   3. 127.  63. 162.  63. 191.  92. 191. 127.]
 [  0.   3. 191. 127. 191. 162. 162. 191. 127. 191.]
 [  0.   3. 127. 191.  92. 191.  63. 162.  63. 127.]
 [  0.   3.  63. 127.  63.  92.  92.  63. 127.  63.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   

In [14]:
# Load and inspect the generated NPY stack
arr = np.load(OUTPUT_FILE)
# print("Loaded shape:", arr.shape)
# print("First sequence (view 0) preview:\n", arr)

np.set_printoptions(threshold=arr.size, linewidth=200)

print("Loaded shape:", arr.shape)
print("Full array:\n", arr)

Loaded shape: (400, 10)
Full array:
 [[  0.   0.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   2. 170.  49.  -1.  -1.  -1.  -1. 139.  49.]
 [  0.   2. 139.  49.  -1.  -1.  -1.  -1. 139. 150.]
 [  0.   2. 139. 150.  -1.  -1.  -1.  -1. 170. 150.]
 [  0.   2. 170. 150.  -1.  -1.  -1.  -1. 170.  49.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.]
 [  0.   

In [16]:
# === BATCH TEST EXPORT CELL: Processes all test samples and saves SVGs/NPYs to svg_vec_new ===

import json
from tqdm import tqdm
# Paths
split_json = r'c:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/train_val_test_split.json'
svg_raw_root = r'c:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_raw'
svg_vec_new_root = r'c:/Users/LEGION/Desktop/cad_project/DeepCAD/data2/svg_vec_new'

# Load test split
with open(split_json, 'r') as f:
    split = json.load(f)
test_ids = split['test']

# Function to process and save in svg_vec_new
def process_test_sample(test_id):
    subdir, sample = test_id.split('/')
    input_dir = os.path.join(svg_raw_root, subdir, sample)
    output_dir = os.path.join(svg_vec_new_root, subdir, sample)
    os.makedirs(output_dir, exist_ok=True)
    # Output file path (now inside the sample folder)
    output_npy = os.path.join(output_dir, sample + '.npy')
    # Copy SVGs and rotate iso
    src_front = os.path.join(input_dir, f'{sample}_Front.svg')
    src_top = os.path.join(input_dir, f'{sample}_Top.svg')
    src_right = os.path.join(input_dir, f'{sample}_Right.svg')
    src_iso = os.path.join(input_dir, f'{sample}_FrontTopRight.svg')
    final_front = os.path.join(output_dir, f'{sample}_Front_final.svg')
    final_right = os.path.join(output_dir, f'{sample}_Right_final.svg')
    final_top = os.path.join(output_dir, f'{sample}_Top_final.svg')
    final_iso = os.path.join(output_dir, f'{sample}_FrontTopRight_final.svg')
    # Rotate iso
    create_rotated_iso_file(src_iso, final_iso, angle=120)
    # Copy others
    for src, dst in [(src_front, final_front), (src_right, final_right), (src_top, final_top)]:
        if os.path.exists(src):
            shutil.copy(src, dst)
    # Generate NPY
    processing_queue = [
        ('Front', final_front, 0),
        ('Right', final_right, 1),
        ('Top', final_top, 2),
        ('Iso', final_iso, 3)
    ]
    all_sequences = []
    for name, fpath, label in processing_queue:
        if not os.path.exists(fpath):
            raw_seq = None
        else:
            raw_seq = convert_svg_to_sequence(fpath, label)
        padded_seq = pad_sequence(raw_seq, label)
        all_sequences.append(padded_seq)
    try:
        final_stack = np.vstack(all_sequences)
        np.save(output_npy, final_stack)
    except Exception as e:
        print(f'Error saving {output_npy}: {e}')

# Process all test samples
for test_id in tqdm(test_ids):
    process_test_sample(test_id)


100%|██████████| 7881/7881 [21:09<00:00,  6.21it/s]
